# Combine NS+OPS and extra_data

This example illustrates how to:

- Load data from two different instruments (NS and OPS)
- Combine them on a common time base
- Inspect and use the `extra_data` tables

In [ ]:
import matplotlib.pyplot as plt
import aerosoltools as at

## Load NS and OPS datasets

A sample NanoScan (NS) and several OPS files are included in the test data directory. Here we utilize the very efficient `load_data_from_folder` function, which can load and combine multiple files from the same instrument, by specifying a path and the loader function to use. In this example we also specify a (optional) search_word, which can be used to load only files containing this specific keyword in their name.

In [ ]:
ns_file = "../../tests/data/Combine_example_NS.csv"
ops_path = "../../tests/data"

ns = at.load_ns_file(ns_file, extra_data  =True)
ops = at.load_data_from_folder(ops_path, at.load_ops_file, search_word="Combine_example",extra_data=True)

The `load_data_from_folder` function displays a progress bar so you can see how many files are loaded and the progress. In addition it prints a summary table, listing the files that were loaded, those that were skipped, and a reason as to why these were skipped. 

This enables you to load many files from a folder, which can include from subfolders, even if the folder contains many different files, where a simple search_word cannot distinguish them efficiently.

## Inspect `extra_data` for each instrument

Many loaders store auxiliary variables (e.g. flow, temperature, status flags)
in the `.extra_data` attribute.

In [ ]:
print("NS extra-data columns:", list(ns.extra_data.columns))
print("OPS extra-data columns:", list(ops.extra_data.columns))

# Show a small snippet if available
display(ns.extra_data.head())
display(ops.extra_data.head())

## Combine NS and OPS on a common time grid

The helper function `combine_size_ranges` aligns the two instruments in time and returns a combined dataset suitable for correlation analysis. If the timestamps between the two datasets does not match precisely e.g. differ a few seconds, then the matching can be relaxed via the `match` keyword. This can be set to either nearest, matching nearest datapoint within a tolerance time window. 

If the two datasets do not have the same sampling frequency e.g. one is per minute and the other is every second, the function can be called with `match = "rebin", rebin_freq = "1min"`, in which case the high frequency data is rebinned into 1 min times to match the low frequency data. A larger and common frequency can also be set.

In [ ]:
combined = at.combine_size_ranges(ns, ops, match="nearest",tolerance="30s",)
combined.data

The resulting variable is an aerosol2d class object, with the same functionalities as the raw loaded data. It is thus possible to plot particle data in the broad size range from 10 nm (Nanoscan lower limit) to 10 µm (OPS upper limit).

The two datasets are combined by simply discarding the upper bins of the Nanoscan, while including all of the OPS bins, as the nanoscan is known to be uncertain at upper range of its particle size spectrum.

In [ ]:
combined.plot_timeseries(y_3d=(1,0));
# as some bins have concentrations of 0 cm-3 it is necessary to set the lower limit to 1 via y_3d = (1,0)
# the upper limit of 0 indicates that the maximum value in the data should be used. 